# 02 — Recherche des images NISAR sur Douala (tâche D2)

Objectif : lister les acquisitions NISAR (L-band) qui recouvrent la zone d'étude, **sans rien télécharger**.
Résultat : un tableau `inventaire_nisar_douala.csv` (date, produit, orbite, polarisation, version, taille).

Ce tableau sert ensuite à **D3** (choix des dates) et **D4** (produit et polarisations).
La recherche ne demande pas de compte Earthdata ; le téléchargement (D5) en demandera un.

In [ ]:
!pip -q install asf_search pandas shapely

## 1. Paramètres

In [ ]:
import json
import pandas as pd
import asf_search as asf
from shapely import wkt as shapely_wkt
from shapely.geometry import shape

# Zone d'étude (D1) : cercle centre 4.02295 N / 9.60535 E, rayon 36 965 m.
# Par défaut : emprise rectangulaire du cercle (simple et fiable pour la recherche).
AOI_WKT = "POLYGON((9.2721 3.69052, 9.9386 3.69052, 9.9386 4.35538, 9.2721 4.35538, 9.2721 3.69052))"

# Option : si vous avez importé douala_aoi.geojson dans Colab (panneau Fichiers), on l'utilise à la place.
import os
if os.path.exists("douala_aoi.geojson"):
    with open("douala_aoi.geojson", encoding="utf-8") as fh:
        AOI_WKT = shape(json.load(fh)["features"][0]["geometry"]).wkt
    print("Zone lue depuis douala_aoi.geojson")

# BETA (oct. 2025 - janv. 2026), PROVISIONAL (juin - oct. 2026), VALIDATED (depuis le 21 oct. 2026)
START = "2025-08-01"
END   = "2026-11-15"

aoi = shapely_wkt.loads(AOI_WKT)
print("Surface de la zone (deg2) :", round(aoi.area, 4))

## 2. Recherche (aucun téléchargement)

In [ ]:
results = asf.geo_search(
    platform=asf.PLATFORM.NISAR,
    intersectsWith=AOI_WKT,
    start=START,
    end=END,
    maxResults=2000,
)
print(f"{len(results)} acquisitions NISAR trouvées sur la zone")

Si le résultat est **0**, essayez d'élargir `START`/`END`, puis vérifiez sur
https://search.asf.alaska.edu (filtre *Dataset = NISAR*) avec la même zone.

In [ ]:
# Découvrir les champs disponibles (les noms exacts peuvent varier selon la version du produit)
if results:
    props = results[0].properties
    for k in sorted(props):
        print(f"{k:28} {str(props[k])[:80]}")

## 3. Tableau d'inventaire

In [ ]:
def first(d, *keys):
    for k in keys:
        v = d.get(k)
        if v not in (None, ""):
            return v
    return None

rows = []
for r in results:
    p = r.properties
    geom = shape(r.geometry) if r.geometry else None
    cover = None
    if geom is not None and geom.is_valid:
        cover = round(100 * geom.intersection(aoi).area / aoi.area, 1)
    rows.append({
        "date":        (first(p, "startTime") or "")[:10],
        "start_time":  first(p, "startTime"),
        "scene":       first(p, "sceneName", "fileID", "granuleUR"),
        "produit":     first(p, "processingLevel", "productType", "beamModeType"),
        "orbite_dir":  first(p, "flightDirection"),
        "piste":       first(p, "pathNumber", "track", "relativeOrbit"),
        "frame":       first(p, "frameNumber", "frame"),
        "polarisation":first(p, "polarization"),
        "version":     first(p, "processingDate", "crid", "collectionName"),
        "taille_MB":   first(p, "bytes", "sizeMB"),
        "couverture_zone_%": cover,
        "url":         first(p, "url"),
    })

df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
df.head(20)

## 4. Résumés utiles pour choisir les dates (D3)

In [ ]:
if df.empty:
    print("Tableau vide : rien à résumer.")
else:
    print("Période couverte :", df["date"].min(), "->", df["date"].max())
    print()
    for titre, col in [("Par produit", "produit"), ("Par direction d'orbite", "orbite_dir"), ("Par piste", "piste")]:
        print(titre, ":")
        print(df[col].value_counts(dropna=False))
        print()
    print("Par mois :");                  print(df.groupby(df["date"].str[:7]).size())

In [ ]:
# Pour D3 : meme piste + meme direction => images comparables. Dates par piste :
if not df.empty:
    for (piste, d), g in df.groupby(["piste", "orbite_dir"], dropna=False):
        dates = sorted(set(g["date"]))
        print(f"piste {piste} / {d} : {len(dates)} dates, couverture moy. {g['couverture_zone_%'].mean():.0f} %")
        print("   ", ", ".join(dates))

## 5. Sauvegarde

In [ ]:
df.to_csv("inventaire_nisar_douala.csv", index=False)
print("Enregistré : inventaire_nisar_douala.csv (panneau Fichiers de Colab, à gauche)")

# Optionnel : copie sur le Drive partagé
# from google.colab import drive
# drive.mount("/content/drive")
# df.to_csv("/content/drive/MyDrive/<dossier_partage>/inventaire_nisar_douala.csv", index=False)

## Fini quand (D2)
- [ ] Le tableau liste les images disponibles avec leur **date**
- [ ] `inventaire_nisar_douala.csv` est partagé avec l'équipe
- [ ] Le nombre de dates par piste est noté pour la réunion D3

**Plan B (D6)** : si 0 résultat ou trop peu de dates, chercher Sentinel-1 (`asf.PLATFORM.SENTINEL1`) sur la même zone.